# Lesson 23 — Conversation Management API

## What you will learn
- Build a **full Conversation Management Layer** (the P5 Chatbot API)
- **Session management**: create, get, list sessions per tenant
- **Usage limits**: message count, session count, RPM rate limiting
- **Agent routing**: route sessions to data_analysis, db_retrieval, or general agent
- **FastAPI endpoints**: REST + SSE streaming

## Architecture (from system diagram)
```
Client
  ↓  POST /sessions          → create session
  ↓  POST /chat              → send message → route to agent
  ↓  GET  /sessions/{id}/history → get history
  ↓  POST /chat/stream       → SSE streaming
FastAPI
  ↓
Session Store (dict → Redis in prod)
  ↓
Usage Limits check
  ↓
LangGraph Orchestrator → routes to specialist
```

## Run the actual server
```bash
uvicorn lesson_23_conversation_api.lesson_23_conversation_api:app --reload --port 8023
```

In [ ]:
# !pip install fastapi uvicorn langchain-ollama langgraph pyjwt

## Step 1 — Session Data Model

In [ ]:
import time
import uuid
import jwt
from dataclasses import dataclass, field
from typing import Optional, Annotated
from typing_extensions import TypedDict
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver

JWT_SECRET = "lesson23-demo"
MAX_MESSAGES = int("50")    # per session
MAX_SESSIONS = int("10")    # per tenant
RATE_LIMIT_RPM = int("20") # requests per minute per user

@dataclass
class Session:
    session_id:   str
    tenant_id:    str
    user_id:      str
    agent_type:   str   # "data_analysis" | "db_retrieval" | "general"
    created_at:   float = field(default_factory=time.time)
    last_active:  float = field(default_factory=time.time)
    message_count: int = 0

# Session store (dict = dev; Redis = production)
_sessions: dict[str, Session] = {}
_request_log: dict[str, list] = {}  # user_id → list of timestamps

def create_demo_token(user_id: str, tenant_id: str, role: str = "user") -> str:
    return jwt.encode(
        {"sub": user_id, "tenant_id": tenant_id, "role": role, "exp": int(time.time()) + 3600},
        JWT_SECRET, algorithm="HS256"
    )

def verify_jwt_token(token: str) -> dict:
    try:
        return {"valid": True, "payload": jwt.decode(token, JWT_SECRET, algorithms=["HS256"])}
    except Exception as e:
        return {"valid": False, "error": str(e)}

print("Session model ready")

## Step 2 — Session Management Functions

In [ ]:
def create_session(user_id: str, tenant_id: str, agent_type: str = "general") -> Session:
    session_id = str(uuid.uuid4())
    session = Session(session_id=session_id, tenant_id=tenant_id, user_id=user_id, agent_type=agent_type)
    _sessions[session_id] = session
    return session

def get_session(session_id: str) -> Optional[Session]:
    return _sessions.get(session_id)

def update_session_activity(session_id: str):
    if s := _sessions.get(session_id):
        s.last_active = time.time()
        s.message_count += 1

def list_sessions_for_tenant(tenant_id: str) -> list[Session]:
    return [s for s in _sessions.values() if s.tenant_id == tenant_id]

def check_usage_limits(session: Session, user_id: str) -> tuple[bool, str]:
    """Returns (allowed, reason_if_denied)."""
    # Limit 1: message count
    if session.message_count >= MAX_MESSAGES:
        return False, f"Session message limit reached ({MAX_MESSAGES})"
    # Limit 2: tenant session count
    tenant_sessions = list_sessions_for_tenant(session.tenant_id)
    if len(tenant_sessions) > MAX_SESSIONS:
        return False, f"Tenant session limit reached ({MAX_SESSIONS})"
    # Limit 3: RPM rate limit
    now = time.time()
    _request_log.setdefault(user_id, [])
    _request_log[user_id] = [t for t in _request_log[user_id] if now - t < 60]
    if len(_request_log[user_id]) >= RATE_LIMIT_RPM:
        return False, f"Rate limit exceeded ({RATE_LIMIT_RPM} RPM)"
    _request_log[user_id].append(now)
    return True, ""

# Test
s = create_session("alice", "acme", "general")
allowed, reason = check_usage_limits(s, "alice")
print(f"Session created: {s.session_id[:8]}... | Allowed: {allowed}")

## Step 3 — LangGraph Orchestrator

In [ ]:
llm = ChatOllama(model="llama3", temperature=0)

class OrchestratorState(TypedDict):
    messages:   Annotated[list, add_messages]
    agent_type: str
    session_id: str

def route_to_agent(state: OrchestratorState) -> str:
    agent_type = state["agent_type"]
    if agent_type in {"data_analysis", "db_retrieval", "general"}:
        return agent_type
    # Keyword fallback
    question = state["messages"][-1].content.lower() if state["messages"] else ""
    if any(w in question for w in ["data", "csv", "excel", "analyze"]):
        return "data_analysis"
    if any(w in question for w in ["database", "sql", "table", "query"]):
        return "db_retrieval"
    return "general"

def data_analysis_node(state: OrchestratorState) -> dict:
    response = llm.invoke([SystemMessage(content="You are a data analysis expert. Be concise."), *state["messages"]])
    return {"messages": [response]}

def db_retrieval_node(state: OrchestratorState) -> dict:
    response = llm.invoke([SystemMessage(content="You are a database expert. Answer SQL/DB questions."), *state["messages"]])
    return {"messages": [response]}

def general_node(state: OrchestratorState) -> dict:
    response = llm.invoke([SystemMessage(content="You are a helpful general assistant."), *state["messages"]])
    return {"messages": [response]}

checkpointer = MemorySaver()
orch_builder = StateGraph(OrchestratorState)
orch_builder.add_node("data_analysis", data_analysis_node)
orch_builder.add_node("db_retrieval",  db_retrieval_node)
orch_builder.add_node("general",       general_node)
orch_builder.add_conditional_edges(START, route_to_agent, {
    "data_analysis": "data_analysis",
    "db_retrieval":  "db_retrieval",
    "general":       "general",
})
for node in ["data_analysis", "db_retrieval", "general"]:
    orch_builder.add_edge(node, END)
orchestrator = orch_builder.compile(checkpointer=checkpointer)
print("Orchestrator compiled!")

## Step 4 — Chat Function (the core /chat endpoint logic)

In [ ]:
def chat(session_id: str, message: str, token: str) -> dict:
    # 1. Auth
    auth = verify_jwt_token(token)
    if not auth["valid"]:
        return {"error": "Unauthorized"}
    user_id = auth["payload"]["sub"]
    
    # 2. Get session
    session = get_session(session_id)
    if not session:
        return {"error": "Session not found"}
    
    # 3. Usage limits
    allowed, reason = check_usage_limits(session, user_id)
    if not allowed:
        return {"error": reason}
    
    # 4. Invoke orchestrator
    config = {"configurable": {"thread_id": session_id}}
    result = orchestrator.invoke({
        "messages":   [HumanMessage(content=message)],
        "agent_type": session.agent_type,
        "session_id": session_id,
    }, config=config)
    
    # 5. Update session metadata
    update_session_activity(session_id)
    
    return {"answer": result["messages"][-1].content, "session_id": session_id}

# Demo
token = create_demo_token("alice", "acme")

# Create sessions for different agent types
s_general = create_session("alice", "acme", "general")
s_db      = create_session("alice", "acme", "db_retrieval")

print("=== General session ===")
r = chat(s_general.session_id, "What is Python used for?", token)
print(f"A: {r.get('answer', r.get('error', ''))[:150]}")

print("\n=== DB session ===")
r = chat(s_db.session_id, "How do I write a GROUP BY query?", token)
print(f"A: {r.get('answer', r.get('error', ''))[:150]}")

## Step 5 — FastAPI Endpoint Overview

The actual API is in `lesson_23_conversation_api.py`. Here's the structure.

In [ ]:
API_ENDPOINTS = """
POST   /sessions                  → create session
POST   /chat                      → send message, get response
GET    /sessions/{id}/history     → full message history
GET    /sessions                  → list all sessions for tenant
POST   /chat/stream               → SSE streaming response
POST   /upload/{session_id}       → upload document for RAG
GET    /health                    → ALB health check
"""
print("Available endpoints:")
print(API_ENDPOINTS)

print("\nTo test the actual server:")
print("  uvicorn lesson_23_conversation_api.lesson_23_conversation_api:app --reload --port 8023")
print()
print("  curl -X POST http://localhost:8023/sessions \\")
print('       -H "Authorization: Bearer $TOKEN" \\') 
print('       -d \'{"agent_type": "general"}\'')

## Key Takeaways

| Layer | Implementation |
|-------|---------------|
| Session store | `dict[session_id, Session]` → Redis in production |
| Auth | JWT decode at every endpoint |
| Usage limits | 3 layers: message count, session count, RPM |
| Agent routing | `session.agent_type` → conditional edges |
| Streaming | `graph.astream()` → `StreamingResponse` with SSE |

## 🏋️ Exercise
1. Start the server and test all endpoints with curl or Postman
2. Implement `delete_session(session_id)` that removes the session AND erases its S3 conversation history (use L22 pattern)
3. Add session expiry: in `list_sessions_for_tenant()`, auto-delete sessions inactive for > 24 hours

In [ ]:
# Your exercise solution here
